# 얼굴 표정 기반 감정 인식 프로젝트 — 처음부터 다시 학습하기

이 노트북은 **기존 코드를 복사해서 이어가는 방식이 아니라, 프로젝트를 처음부터 다시 이해하면서 작성하는 학습용 기준 파일**입니다.

## 이번 재시작 원칙

- **수업에서 다루지 않은 고급 기법은 넣지 않습니다.**
- 7개 감정 클래스는 그대로 사용합니다.
- Training 데이터는 현재 프로젝트 결정대로 **TRAIN_01 + TRAIN_02**를 기준으로 합니다.
- EDA에서 확인했던 **다수결 감정 라벨 생성, 중앙값 Bounding Box 생성은 사용하지 않습니다.**
- 얼굴 crop, 흑백 변환 같은 처리는 처음부터 자동 적용하지 않습니다.
- 먼저 **원본 데이터 → 표 만들기 → 확인 → 분할 → 이미지 읽기/resize → 기본 CNN 학습** 순서로 갑니다.
- 처음에는 작은 데이터로 코드가 정상 동작하는지 확인한 뒤, 전체 학습으로 늘립니다.

> 목표: 코드를 '돌리는 것'보다 **각 단계가 왜 필요한지 설명할 수 있게 만드는 것**


## 0. 프로젝트 흐름

이번 파일에서는 아래 순서만 사용합니다.

1. 필요한 라이브러리 불러오기  
2. 프로젝트 경로 확인  
3. Training 이미지 목록 만들기  
4. 7개 클래스 분포 확인  
5. 중복 / 결측 / 파일 존재 여부 확인  
6. 학습용 DataFrame 만들기  
7. `train_test_split()`으로 학습/검증 데이터 나누기  
8. OpenCV로 이미지 읽기  
9. resize와 픽셀값 정규화  
10. PyTorch Dataset / DataLoader 만들기  
11. 가장 기본적인 CNN 모델 만들기  
12. 학습 결과 확인

처음 실행할 때는 **샘플 학습**으로 설정해 둡니다.


In [ ]:
# 1. 필요한 라이브러리 불러오기
# 수업에서 사용한 Python / Pandas / NumPy / OpenCV / scikit-learn / PyTorch 범위로 구성합니다.

import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### 꼭 이해할 내용

- `os` : 폴더와 파일 경로를 확인할 때 사용
- `json` : AI-Hub 라벨 JSON을 읽을 때 사용
- `pandas` : 이미지 경로와 감정 라벨을 표 형태로 관리
- `cv2` : 이미지 읽기, 크기 변경
- `train_test_split` : 학습용 / 검증용 데이터 분리
- ``torch`, `torch.nn` : 기본 CNN 구성


In [ ]:
# 2. 프로젝트 기본 경로 설정
# 옆자리 PC에서도 같은 구조를 사용할 수 있도록 프로젝트 폴더만 한 곳에서 지정합니다.

PROJECT_DIR = r"D:\emotion_recognition_project"

TRAIN_IMAGE_DIR = os.path.join(PROJECT_DIR, "raw", "train")
TRAIN_LABEL_DIR = os.path.join(PROJECT_DIR, "labels", "train")

print("프로젝트 폴더:", PROJECT_DIR)
print("Training 이미지 폴더:", TRAIN_IMAGE_DIR)
print("Training 라벨 폴더:", TRAIN_LABEL_DIR)

print("\n[폴더 존재 여부]")
print("PROJECT_DIR:", os.path.exists(PROJECT_DIR))
print("TRAIN_IMAGE_DIR:", os.path.exists(TRAIN_IMAGE_DIR))
print("TRAIN_LABEL_DIR:", os.path.exists(TRAIN_LABEL_DIR))


## 3. 먼저 이미지 파일 자체를 확인한다

복잡한 라벨 가공부터 하지 않고, **실제로 사용할 TRAIN_01·TRAIN_02 이미지가 폴더에 존재하는지**부터 확인합니다.

현재 프로젝트에서 확인한 기준 수량은 **223,578장**입니다.  
아래 코드는 폴더 안의 `.jpg`, `.jpeg` 파일만 찾습니다.


In [ ]:
# 3-1. Training 이미지 파일 목록 만들기

image_paths = []

for root, dirs, files in os.walk(TRAIN_IMAGE_DIR):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg")):
            full_path = os.path.join(root, file)
            image_paths.append(full_path)

print("찾은 Training 이미지 수:", len(image_paths))

# 앞의 5개 경로만 확인
for path in image_paths[:5]:
    print(path)


## 4. 이미지 경로에서 현재 폴더 클래스 확인

이번 재시작 파일에서는 **처음부터 다수결 라벨을 만들지 않습니다.**

우선 Training 폴더가 감정별로 정리되어 있다는 전제에서,
이미지가 들어 있는 **상위 폴더명**을 `source_emotion`으로 확인합니다.

> 이 단계는 '최종 정답 라벨을 새로 만드는 과정'이 아니라  
> 현재 저장된 데이터가 어떤 감정 폴더에 들어 있는지 구조를 확인하는 단계입니다.


In [ ]:
# 4-1. 이미지 파일명 / 경로 / 폴더 감정을 DataFrame으로 정리

rows = []

for path in image_paths:
    filename = os.path.basename(path)
    source_emotion = os.path.basename(os.path.dirname(path))

    rows.append({
        "filename": filename,
        "image_path": path,
        "source_emotion": source_emotion
    })

image_df = pd.DataFrame(rows)

print("DataFrame 크기:", image_df.shape)
display(image_df.head())


In [ ]:
# 4-2. 클래스 이름과 이미지 개수 확인

emotion_counts = image_df["source_emotion"].value_counts().sort_index()

print(emotion_counts)
print("
클래스 수:", image_df["source_emotion"].nunique())


In [ ]:
# 4-3. 클래스 분포 시각화

emotion_counts.plot(kind="bar")
plt.title("Training Image Count by Emotion")
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


## 5. 가장 기본적인 데이터 품질 확인

EDA에서 했던 복잡한 가공을 반복하지 않습니다.  
이번에는 수업 범위에서 이해해야 할 기본 확인만 다시 합니다.

- 행 개수
- 중복 파일명
- 결측치
- 클래스별 개수


In [ ]:
# 5-1. 기본 정보 확인

print("전체 이미지 수:", len(image_df))
print("중복 filename 수:", image_df["filename"].duplicated().sum())

print("
결측치 개수")
print(image_df.isnull().sum())


## 6. 이미지가 실제로 열리는지 직접 확인

바로 모델 학습으로 넘어가지 않고, 먼저 OpenCV로 이미지 1장을 읽습니다.

OpenCV의 이미지 shape은 보통:

`(높이, 너비, 채널)`

형태입니다.


In [ ]:
# 6-1. 이미지 1장 읽기

sample_path = image_df.iloc[0]["image_path"]

sample_img = cv2.imread(sample_path)

print("샘플 이미지 경로:", sample_path)
print("이미지 shape:", sample_img.shape)
print("데이터 타입:", sample_img.dtype)


In [ ]:
# 6-2. OpenCV는 BGR 순서로 읽기 때문에 화면 출력용으로 RGB로 변환

sample_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(sample_rgb)
plt.title(image_df.iloc[0]["source_emotion"])
plt.axis("off")
plt.show()


## 7. 학습/검증 데이터 분리

`train_test_split()`을 사용합니다.

- `test_size=0.2` : 80% 학습, 20% 검증
- `random_state=42` : 실행할 때마다 같은 방식으로 나누기
- `stratify=y` : 7개 감정 클래스 비율을 비슷하게 유지

이번에는 **별도의 복잡한 샘플링 기법은 사용하지 않습니다.**


In [ ]:
# 7-1. X와 y 준비

X = image_df["image_path"]
y = image_df["source_emotion"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습 데이터 수:", len(X_train))
print("검증 데이터 수:", len(X_valid))


In [ ]:
# 7-2. 분할 후 클래스 비율 확인

print("[학습 데이터 클래스 분포]")
print(y_train.value_counts(normalize=True).sort_index())

print("\n[검증 데이터 클래스 분포]")
print(y_valid.value_counts(normalize=True).sort_index())


## 8. 처음에는 작은 샘플로 학습한다

223,578장을 바로 학습시키기 전에 코드가 정상적으로 동작하는지 확인합니다.

처음부터 전체 데이터를 돌려서 몇 시간 뒤 오류를 발견하는 것을 막기 위한 단계입니다.

아래 기본값은 **클래스당 최대 500장**입니다.  
코드가 정상 동작한 뒤 `USE_SAMPLE = False`로 바꾸면 전체 데이터를 대상으로 준비할 수 있습니다.


In [ ]:
# 8-1. 분할 결과를 다시 DataFrame으로 묶기

train_df = pd.DataFrame({
    "image_path": X_train.values,
    "emotion": y_train.values
})

valid_df = pd.DataFrame({
    "image_path": X_valid.values,
    "emotion": y_valid.values
})

USE_SAMPLE = True
SAMPLE_PER_CLASS = 500

if USE_SAMPLE:
    train_df = (
        train_df.groupby("emotion", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_CLASS), random_state=42))
        .reset_index(drop=True)
    )

    valid_df = (
        valid_df.groupby("emotion", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), 100), random_state=42))
        .reset_index(drop=True)
    )

print("현재 학습에 사용할 데이터 수:", len(train_df))
print("현재 검증에 사용할 데이터 수:", len(valid_df))
print(train_df["emotion"].value_counts().sort_index())


## 9. 감정 라벨을 숫자로 바꾸기

딥러닝 모델은 문자열 감정명을 그대로 계산하지 못하므로  
`0 ~ 6`의 숫자로 변환합니다.

이 단계에서는 직접 딕셔너리를 만들어 **라벨이 어떻게 숫자로 바뀌는지 눈으로 확인**합니다.


In [ ]:
# 9-1. 클래스 이름 정렬 후 번호 부여

class_names = sorted(train_df["emotion"].unique())

label_to_index = {}

for index, label in enumerate(class_names):
    label_to_index[label] = index

print("클래스 목록:", class_names)
print("라벨 번호:", label_to_index)

train_df["label"] = train_df["emotion"].map(label_to_index)
valid_df["label"] = valid_df["emotion"].map(label_to_index)

display(train_df.head())


## 10. 이미지 전처리 함수

이번에는 수업에서 이해하기 쉬운 최소 전처리만 적용합니다.

1. 이미지 읽기
2. BGR → RGB
3. `128 × 128` resize
4. 픽셀값을 `0~1` 범위로 변환

**얼굴 Bounding Box crop, 흑백 변환, 복잡한 증강은 아직 적용하지 않습니다.**


In [ ]:
# 10-1. 이미지 크기 설정

IMG_SIZE = 128

def load_image(image_path):
    # 이미지 읽기
    img = cv2.imread(image_path)

    # 혹시 이미지가 열리지 않으면 오류 발생
    if img is None:
        raise ValueError("이미지를 읽을 수 없습니다: " + image_path)

    # BGR -> RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 크기 통일
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # 0~255 -> 0~1
    img = img.astype(np.float32) / 255.0

    return img


In [ ]:
# 10-2. 전처리 결과 1장 확인

test_img = load_image(train_df.iloc[0]["image_path"])

print("shape:", test_img.shape)
print("dtype:", test_img.dtype)
print("최솟값:", test_img.min())
print("최댓값:", test_img.max())

plt.imshow(test_img)
plt.title(train_df.iloc[0]["emotion"])
plt.axis("off")
plt.show()


## 11. PyTorch Dataset과 DataLoader 만들기

PyTorch에서는 이미지를 전부 한 번에 배열로 만들기보다, `Dataset`과 `DataLoader`를 사용해 필요한 만큼 배치 단위로 불러오는 방식으로 학습합니다.


In [ ]:
# 11-1. PyTorch Dataset 정의

class EmotionDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.loc[index, "image_path"]
        label = self.dataframe.loc[index, "label"]

        img = load_image(image_path)

        # PyTorch는 이미지 순서를 (높이, 너비, 채널)이 아니라
        # (채널, 높이, 너비) 형태로 사용합니다.
        img = np.transpose(img, (2, 0, 1))

        img = torch.tensor(img, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        return img, label


train_dataset = EmotionDataset(train_df)
valid_dataset = EmotionDataset(valid_df)

print("train_dataset:", len(train_dataset))
print("valid_dataset:", len(valid_dataset))


In [ ]:
# 11-2. DataLoader 만들기

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

images, labels = next(iter(train_loader))

print("배치 이미지 shape:", images.shape)
print("배치 라벨 shape:", labels.shape)


## 12. 가장 기본적인 PyTorch CNN 모델

이번 재시작에서는 먼저 **전이학습 모델을 사용하지 않고 기본 CNN 구조**부터 이해합니다.

흐름:

`Conv2d → MaxPool2d → Conv2d → MaxPool2d → Flatten → Linear → 출력`

7개 감정 중 하나를 고르는 **다중 분류 문제**이므로 마지막 출력 노드는 7개입니다.


In [ ]:
# 12-1. 기본 CNN 모델 만들기

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN(num_classes=len(class_names)).to(device)

print(model)
print("사용 장치:", device)


## 13. 모델 학습 설정

- `CrossEntropyLoss()` : 0~6 정수 라벨을 사용하는 다중 분류 손실 함수
- `Adam` : 가중치를 수정하는 최적화 방법
- accuracy : 전체 중 맞춘 비율


In [ ]:
# 13-1. 손실 함수와 옵티마이저 설정

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


In [ ]:
# 13-2. 샘플 학습
# 처음에는 epoch를 작게 두고 전체 흐름이 정상 작동하는지 먼저 확인합니다.

EPOCHS = 5

train_acc_history = []
valid_acc_history = []
train_loss_history = []
valid_loss_history = []

for epoch in range(EPOCHS):
    # -------------------------
    # 1) 학습 단계
    # -------------------------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # -------------------------
    # 2) 검증 단계
    # -------------------------
    model.eval()

    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            valid_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            valid_correct += (predictions == labels).sum().item()
            valid_total += labels.size(0)

    valid_loss /= valid_total
    valid_acc = valid_correct / valid_total

    train_loss_history.append(train_loss)
    valid_loss_history.append(valid_loss)
    train_acc_history.append(train_acc)
    valid_acc_history.append(valid_acc)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_loss={valid_loss:.4f} | "
        f"val_acc={valid_acc:.4f}"
    )


## 14. 학습 결과 그래프 확인

학습 정확도만 보지 말고 **validation 결과가 같이 움직이는지** 확인합니다.


In [ ]:
# 14-1. Accuracy 그래프

plt.plot(train_acc_history, label="train_accuracy")
plt.plot(valid_acc_history, label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
# 14-2. Loss 그래프

plt.plot(train_loss_history, label="train_loss")
plt.plot(valid_loss_history, label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


# 여기까지 먼저 이해할 것

## 반드시 설명할 수 있어야 하는 코드 5개

1. `pd.DataFrame(rows)`  
   → 이미지 경로와 감정 정보를 표로 만든다.

2. `value_counts()`  
   → 각 감정 클래스의 데이터 개수를 확인한다.

3. `train_test_split(..., stratify=y)`  
   → 학습/검증 데이터를 나누면서 클래스 비율을 유지한다.

4. `cv2.resize()` + `/ 255.0`  
   → 이미지 크기를 통일하고 픽셀값 범위를 줄인다.

5. `Conv2d → MaxPool2d → Linear`  
   → 이미지 특징을 추출하고 최종 감정을 분류한다.

---

## 이번 파일에서 의도적으로 제외한 것

- annot_A/B/C **다수결 라벨 생성**
- A/B/C Bounding Box **중앙값 계산**
- 얼굴 자동 crop을 최종 규칙으로 적용
- 흑백 변환
- 복잡한 데이터 증강
- ResNet / EfficientNet 같은 전이학습 모델
- 별도의 커스텀 데이터 제너레이터
- 수업에서 아직 배우지 않은 성능 개선 기법

이것들은 **기본 흐름을 이해한 뒤, 실제 모델 비교 단계에서 필요한 것만 추가**합니다.

---

## 선생님이 물어볼 가능성이 높은 질문

- 왜 TRAIN_01·02만 사용했나요?
- 왜 7개 클래스를 유지했나요?
- `stratify=y`는 왜 넣었나요?
- resize가 왜 필요한가요?
- 픽셀값을 255로 나누는 이유는 무엇인가요?
- CNN에서 Conv2d와 MaxPool2d는 각각 무슨 역할인가요?
- train accuracy와 validation accuracy가 크게 차이나면 무엇을 의심해야 하나요?

---

## 직접 설명해보기

**Q. 지금 프로젝트의 전체 흐름을 30초 안에 설명한다면?**

> 원본 이미지의 경로와 감정 클래스를 DataFrame으로 정리하고, 중복과 결측 및 클래스 분포를 확인했습니다. 이후 train_test_split을 이용해 학습과 검증 데이터를 나누고, OpenCV로 이미지를 읽어 같은 크기로 resize한 뒤 픽셀값을 정규화했습니다. 먼저 기본 CNN으로 7개 감정 분류가 정상적으로 학습되는지 확인하고, 이후 결과를 기준으로 필요한 전처리와 모델 비교를 진행할 예정입니다.
